<a href="https://colab.research.google.com/github/lcato061/AB-testing/blob/main/A_B_test_button_color.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A/B Testing Button Color: Product Decision Analysis

## Objective
Evaluate whether changing a button from gray (control) to green (treatment) improves conversion rates.

## Business Context
Button color changes are often used to increase conversions, but even small changes can impact user behavior.

Goal: Determine whether the new design should be launched.

## Hypothesis

- H0 (null): Conversion rate is the same between control and treatment
- H1 (alternative): Treatment (green button) increases conversion rate

In [ ]:
from google.colab import files

# Open a file upload dialog
uploaded = files.upload()

# Access the file
import pandas as pd
import io

df = pd.read_csv(io.BytesIO(uploaded['ab_data (1).csv']))
print(df.head())

Saving ab_data.csv to ab_data (1).csv
   user_id                   timestamp      group landing_page  converted
0   851104  2017-01-21 22:11:48.556739    control     old_page          0
1   804228  2017-01-12 08:01:45.159739    control     old_page          0
2   661590  2017-01-11 16:55:06.154213  treatment     new_page          0
3   853541  2017-01-08 18:28:03.143765  treatment     new_page          0
4   864975  2017-01-21 01:52:26.210827    control     old_page          1


## Data Cleaning

- Removed duplicate user_ids to ensure independence of observations
- Kept first instance of duplicated user_ids and dropped others
- Verified no missing values
- Ensured balanced group sizes

In [ ]:
df.describe()

,user_id,converted
count,294478.000000,294478.000000
mean,787974.124733,0.119659
std,91210.823776,0.324563
min,630000.000000,0.000000
25%,709032.250000,0.000000
50%,787933.500000,0.000000
75%,866911.750000,0.000000
max,945999.000000,1.000000


In [ ]:
df.info()
df.isna().sum()
df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294478 non-null  int64 
 1   timestamp     294478 non-null  object
 2   group         294478 non-null  object
 3   landing_page  294478 non-null  object
 4   converted     294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


np.int64(0)

In [ ]:
df['converted'].unique()

array([0, 1])

In [ ]:
df['group'].value_counts()
df['landing_page'].value_counts()
df['converted'].value_counts()

,count
converted,
0,259241
1,35237


In [ ]:
duplicates = df[df.duplicated(subset=['user_id'], keep=False)]
print(duplicates)

        user_id                   timestamp      group landing_page  converted
22       767017  2017-01-12 22:58:14.991443    control     new_page          0
192      656468  2017-01-18 07:13:29.805052  treatment     new_page          1
226      773693  2017-01-23 18:05:45.167335    control     old_page          1
240      733976  2017-01-11 15:11:16.407599    control     new_page          0
246      704650  2017-01-04 19:10:52.655062  treatment     new_page          0
...         ...                         ...        ...          ...        ...
294308   905197  2017-01-03 06:56:47.488231  treatment     new_page          0
294309   787083  2017-01-17 00:15:20.950723    control     old_page          0
294328   641570  2017-01-09 21:59:27.695711    control     old_page          0
294331   689637  2017-01-13 11:34:28.339532    control     new_page          0
294355   744456  2017-01-13 09:32:07.106794  treatment     new_page          0

[7788 rows x 5 columns]


In [ ]:
df_unique = df.drop_duplicates(subset='user_id', keep='first')
print(df_unique)

        user_id                   timestamp      group landing_page  converted
0        851104  2017-01-21 22:11:48.556739    control     old_page          0
1        804228  2017-01-12 08:01:45.159739    control     old_page          0
2        661590  2017-01-11 16:55:06.154213  treatment     new_page          0
3        853541  2017-01-08 18:28:03.143765  treatment     new_page          0
4        864975  2017-01-21 01:52:26.210827    control     old_page          1
...         ...                         ...        ...          ...        ...
294473   751197  2017-01-03 22:28:38.630509    control     old_page          0
294474   945152  2017-01-12 00:51:57.078372    control     old_page          0
294475   734608  2017-01-22 11:45:03.439544    control     old_page          0
294476   697314  2017-01-15 01:20:28.957438    control     old_page          0
294477   715931  2017-01-16 12:40:24.467417  treatment     new_page          0

[290584 rows x 5 columns]


In [ ]:
df_unique['group'].value_counts()
df_unique['landing_page'].value_counts()
df_unique['converted'].value_counts()

,count
converted,
0,255839
1,34745


## Results Summary

- Control conversion rate: 12.0%
- Treatment conversion rate: 11.8%
- Difference: -0.2%

- Number of users (gray): 145,352
- Number of users (green): 145,252

In [ ]:
conversion_rate = df_unique.groupby('group')['converted'].mean()
conversion_rate

,converted
group,
control,0.120297
treatment,0.118843


In [ ]:
import numpy as np
old = df_unique[df_unique['group']=='control']
new = df_unique[df_unique['group']=='treatment']

count = np.array([new['converted'].sum(), old['converted'].sum()])
nobs = np.array([len(new), len(old)])
count, nobs

(array([17274, 17471]), array([145352, 145232]))

## Statistical Test

- Z-statistic: -1.21
- P-value: 0.227

### Interpretation
The p-value > 0.05, so we fail to reject the null hypothesis.

There is no statistically significant difference between the two variants.

# Final Decision

We fail to reject the null hypothesis and conclude that there is no evidence of a significant increase in conversion rate between the gray and green button options.

-> Do not implement the green button

### Reasoning:
- No statistically significant improvement in conversion rate
- Observed performance is slightly worse in the treatment group
- No evidence of business benefit

### Business Impact:
Deploying the new design would introduce risk without measurable upside


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Two-proportion z-test
z_stat, p_value = proportions_ztest(count=count, nobs=nobs)
print("Z-statistic:", z_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("✅ Statistically significant difference!")
else:
    print("⚠️ No significant difference.")

Z-statistic: -1.2083846739740718
P-value: 0.22689933216132785
⚠️ No significant difference.


## Next Steps

- Test additional design variations (color, size, placement)
- Run experiment longer to increase statistical power
- Analyze user segments to identify differential effects

Goal: Identify a design that produces a meaningful increase in conversion.

-> Statistical significance alone is not enough—product decisions must also consider business impact and practical significance.